In [5]:
import pandas as pd
import seaborn as sns
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score


# KNN Model

In [6]:
data = pd.read_csv("clean_data.csv")
test = pd.read_csv('clean_test.csv')


## One hot encoding for the model on contract length and gender

In [7]:
df_encoded = pd.get_dummies(data, columns=['Contract Length','Gender'], prefix=['Contract Length','Gender'], dtype=int)


Split the data

In [9]:
#split the data into train and validaiton
X = df_encoded[['Total Spend','Support Calls','Contract Length_Monthly','Last Interaction','Age','Contract Length_Quarterly','Gender_Female']]
y = df_encoded['Churn']
X_train, X_validation, y_train, y_validation = train_test_split(X,y, test_size=.3, random_state=123)


through trial and error I discovered that 11 is the best number of neighbors

In [12]:
knn_spec = KNeighborsClassifier(n_neighbors=11)


knn_fit = knn_spec.fit(X=X_train, y=y_train)

In [13]:
knn_fit.predict_proba(X_train)

array([[0.90909091, 0.09090909],
       [0.81818182, 0.18181818],
       [0.27272727, 0.72727273],
       ...,
       [1.        , 0.        ],
       [0.90909091, 0.09090909],
       [0.90909091, 0.09090909]], shape=(212186, 2))

In [14]:
class_prediction = knn_fit.predict_proba(X_validation)


## How well does my mdoel work?

In [15]:

churn_prob = class_prediction[:,1]

print(roc_auc_score(y_validation,churn_prob))

0.8702458295912889


## Now use this model on my test data!

In [13]:
test_encoded = pd.get_dummies(test, columns=['Contract Length','Gender'], prefix=['Contract Length','Gender'], dtype=int)


X_test=test_encoded[['Total Spend','Support Calls','Contract Length_Monthly','Last Interaction','Age','Contract Length_Quarterly','Gender_Female']]
class_prediction = knn_fit.predict_proba(X_test)
class_prediction[:,1]


array([[0.27272727, 0.72727273],
       [0.72727273, 0.27272727],
       [1.        , 0.        ],
       ...,
       [0.63636364, 0.36363636],
       [0.09090909, 0.90909091],
       [1.        , 0.        ]], shape=(133776, 2))

In [11]:
prob = test[["CustomerID"]].copy()
prob['Churn'] = class_prediction[:,1]
prob['Churn']

0         0.727273
1         0.272727
2         0.000000
3         0.909091
4         0.454545
            ...   
133771    0.181818
133772    0.363636
133773    0.363636
133774    0.909091
133775    0.000000
Name: Churn, Length: 133776, dtype: float64

In [12]:
prob.to_csv("prob_knn_1.csv",index=False)